# Model Analysis and Visualization

This notebook analyzes the trained Siamese network model by:
1. Testing model accuracy
2. Visualizing embeddings using t-SNE
3. Plotting class means in the embedding space

In [12]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
import os

In [13]:
# Contrastive loss
def contrastive_loss(y_true, y_pred, margin=1.0):
    y_true = tf.cast(y_true, y_pred.dtype)
    positive_loss = y_true * tf.square(y_pred)
    negative_loss = (1 - y_true) * tf.square(tf.maximum(margin - y_pred, 0))
    return tf.reduce_mean(positive_loss + negative_loss)

# Custom accuracy
def accuracy(y_true, y_pred, threshold=1.0):
    print(f"{y_pred} < {threshold}")
    y_pred_labels = tf.cast(y_pred < threshold, tf.float32)
    return tf.reduce_mean(tf.cast(tf.equal(y_true, y_pred_labels), tf.float32))


In [14]:
# Load the dataset
data = pd.read_csv("augmented_hand_landmarks.csv")

# Extract features and labels
X = data.drop(columns=['label']).values
y = data['label'].values


# Generate pairs (This example assumes binary labels for simplicity, modify as per your dataset)

def create_pairs(X, y):
    pairs = []
    labels = []
    unique_labels = np.unique(y)
    label_to_indices = {label: np.where(y == label)[0] for label in unique_labels}
    
    for label in unique_labels:
        indices = label_to_indices[label]
        num_indices = len(indices)
        
        # Positive pairs
        for i in range(num_indices):
            # Randomly select another sample from the same class
            j = np.random.choice(indices)
            pairs.append([X[indices[i]], X[j]])
            labels.append(1)  # Similar pair
        
        # Negative pairs
        for i in range(num_indices):
            # Select a different label
            different_label = np.random.choice(unique_labels[unique_labels != label])
            neg_index = np.random.choice(label_to_indices[different_label])
            pairs.append([X[indices[i]], X[neg_index]])
            labels.append(0)  # Dissimilar pair
    
    return np.array(pairs), np.array(labels)

pairs, labels = create_pairs(X, y)


In [15]:
# Load the trained model
siamese_model = tf.keras.models.load_model('embedding_model.h5',
                                          custom_objects={'contrastive_loss': contrastive_loss,
                                                         'accuracy': accuracy})

# Get the base network for embeddings
base_network = Model(inputs=siamese_model.get_layer(index=2).input,
                    outputs=siamese_model.get_layer(index=2).output)

## 1. Model Accuracy Testing

## 2. t-SNE Visualization

In [ ]:
def compute_embeddings(X):
    """Compute embeddings for input data using the base network"""
    return base_network.predict(X)

def compute_accuracy(X, y, threshold=1.0):
    """Compute accuracy using embeddings and nearest neighbor classification"""
    # Get embeddings
    embeddings = compute_embeddings(X)
    
    # For each sample, find the nearest neighbor
    predictions = []
    for i, emb in enumerate(embeddings):
        # Compute distances to all other points
        distances = np.sum((embeddings - emb) ** 2, axis=1)
        distances[i] = np.inf  # Exclude self
        nearest_idx = np.argmin(distances)
        predictions.append(y[nearest_idx])
    
    return accuracy_score(y, predictions)

# Compute and print accuracy
accuracy = compute_accuracy(X, y)
print(f"Model Accuracy: {accuracy:.4f}")

# Compute confusion matrix
embeddings = compute_embeddings(X)
predictions = []
for i, emb in enumerate(embeddings):
    distances = np.sum((embeddings - emb) ** 2, axis=1)
    distances[i] = np.inf
    nearest_idx = np.argmin(distances)
    predictions.append(y[nearest_idx])

cm = confusion_matrix(y, predictions)

# Plot confusion matrix
plt.figure(figsize=(15, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

ValueError: in user code:

    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\input_

In [ ]:
def compute_embeddings(X):
    """Compute embeddings for input data using the base network"""
    return base_network.predict(X)

def compute_accuracy(X, y, threshold=1.0):
    """Compute accuracy using embeddings and nearest neighbor classification"""
    # Get embeddings
    embeddings = compute_embeddings(X)
    
    # For each sample, find the nearest neighbor
    predictions = []
    for i, emb in enumerate(embeddings):
        # Compute distances to all other points
        distances = np.sum((embeddings - emb) ** 2, axis=1)
        distances[i] = np.inf  # Exclude self
        nearest_idx = np.argmin(distances)
        predictions.append(y[nearest_idx])
    
    return accuracy_score(y, predictions)

# Compute and print accuracy
accuracy = compute_accuracy(X, y)
print(f"Model Accuracy: {accuracy:.4f}")

# Compute confusion matrix
embeddings = compute_embeddings(X)
predictions = []
for i, emb in enumerate(embeddings):
    distances = np.sum((embeddings - emb) ** 2, axis=1)
    distances[i] = np.inf
    nearest_idx = np.argmin(distances)
    predictions.append(y[nearest_idx])

cm = confusion_matrix(y, predictions)

# Plot confusion matrix
plt.figure(figsize=(15, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

ValueError: in user code:

    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\input_

In [ ]:
def compute_embeddings(X):
    """Compute embeddings for input data using the base network"""
    return base_network.predict(X)

def compute_accuracy(X, y, threshold=1.0):
    """Compute accuracy using embeddings and nearest neighbor classification"""
    # Get embeddings
    embeddings = compute_embeddings(X)
    
    # For each sample, find the nearest neighbor
    predictions = []
    for i, emb in enumerate(embeddings):
        # Compute distances to all other points
        distances = np.sum((embeddings - emb) ** 2, axis=1)
        distances[i] = np.inf  # Exclude self
        nearest_idx = np.argmin(distances)
        predictions.append(y[nearest_idx])
    
    return accuracy_score(y, predictions)

# Compute and print accuracy
accuracy = compute_accuracy(X, y)
print(f"Model Accuracy: {accuracy:.4f}")

# Compute confusion matrix
embeddings = compute_embeddings(X)
predictions = []
for i, emb in enumerate(embeddings):
    distances = np.sum((embeddings - emb) ** 2, axis=1)
    distances[i] = np.inf
    nearest_idx = np.argmin(distances)
    predictions.append(y[nearest_idx])

cm = confusion_matrix(y, predictions)

# Plot confusion matrix
plt.figure(figsize=(15, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

ValueError: in user code:

    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\input_

In [ ]:
def compute_embeddings(X):
    """Compute embeddings for input data using the base network"""
    return base_network.predict(X)

def compute_accuracy(X, y, threshold=1.0):
    """Compute accuracy using embeddings and nearest neighbor classification"""
    # Get embeddings
    embeddings = compute_embeddings(X)
    
    # For each sample, find the nearest neighbor
    predictions = []
    for i, emb in enumerate(embeddings):
        # Compute distances to all other points
        distances = np.sum((embeddings - emb) ** 2, axis=1)
        distances[i] = np.inf  # Exclude self
        nearest_idx = np.argmin(distances)
        predictions.append(y[nearest_idx])
    
    return accuracy_score(y, predictions)

# Compute and print accuracy
accuracy = compute_accuracy(X, y)
print(f"Model Accuracy: {accuracy:.4f}")

# Compute confusion matrix
embeddings = compute_embeddings(X)
predictions = []
for i, emb in enumerate(embeddings):
    distances = np.sum((embeddings - emb) ** 2, axis=1)
    distances[i] = np.inf
    nearest_idx = np.argmin(distances)
    predictions.append(y[nearest_idx])

cm = confusion_matrix(y, predictions)

# Plot confusion matrix
plt.figure(figsize=(15, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

ValueError: in user code:

    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\input_

In [ ]:
def compute_embeddings(X):
    """Compute embeddings for input data using the base network"""
    return base_network.predict(X)

def compute_accuracy(X, y, threshold=1.0):
    """Compute accuracy using embeddings and nearest neighbor classification"""
    # Get embeddings
    embeddings = compute_embeddings(X)
    
    # For each sample, find the nearest neighbor
    predictions = []
    for i, emb in enumerate(embeddings):
        # Compute distances to all other points
        distances = np.sum((embeddings - emb) ** 2, axis=1)
        distances[i] = np.inf  # Exclude self
        nearest_idx = np.argmin(distances)
        predictions.append(y[nearest_idx])
    
    return accuracy_score(y, predictions)

# Compute and print accuracy
accuracy = compute_accuracy(X, y)
print(f"Model Accuracy: {accuracy:.4f}")

# Compute confusion matrix
embeddings = compute_embeddings(X)
predictions = []
for i, emb in enumerate(embeddings):
    distances = np.sum((embeddings - emb) ** 2, axis=1)
    distances[i] = np.inf
    nearest_idx = np.argmin(distances)
    predictions.append(y[nearest_idx])

cm = confusion_matrix(y, predictions)

# Plot confusion matrix
plt.figure(figsize=(15, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

ValueError: in user code:

    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "d:\kht3327\_Projects\Major FYP\proj\venv\lib\site-packages\keras\engine\input_

In [ ]:
# Compute t-SNE
tsne = TSNE(n_components=2, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

# Plot t-SNE
plt.figure(figsize=(15, 10))
unique_labels = np.unique(y)
colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    mask = y == label
    plt.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                c=[color], label=label, alpha=0.6)

plt.title('t-SNE Visualization of Embeddings')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Class Means Visualization

In [ ]:
# Compute class means in embedding space
class_means = {}
for label in unique_labels:
    mask = y == label
    class_means[label] = np.mean(embeddings[mask], axis=0)

# Convert means to array for t-SNE
means_array = np.array(list(class_means.values()))
means_labels = list(class_means.keys())

# Apply t-SNE to means
means_2d = TSNE(n_components=2, random_state=42).fit_transform(means_array)

# Plot class means
plt.figure(figsize=(15, 10))
colors = plt.cm.rainbow(np.linspace(0, 1, len(means_labels)))

for i, (label, color) in enumerate(zip(means_labels, colors)):
    plt.scatter(means_2d[i, 0], means_2d[i, 1], 
                c=[color], label=label, s=100)
    plt.annotate(label, (means_2d[i, 0], means_2d[i, 1]), 
                 xytext=(5, 5), textcoords='offset points')

plt.title('t-SNE Visualization of Class Means')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Compute and visualize mean distances between classes
n_classes = len(unique_labels)
distance_matrix = np.zeros((n_classes, n_classes))

for i, label1 in enumerate(means_labels):
    for j, label2 in enumerate(means_labels):
        distance = np.sum((class_means[label1] - class_means[label2]) ** 2)
        distance_matrix[i, j] = distance

plt.figure(figsize=(15, 10))
sns.heatmap(distance_matrix, xticklabels=means_labels, 
            yticklabels=means_labels, annot=True, fmt='.2f')
plt.title('Distance Matrix between Class Means')
plt.show()